In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name
import json

source_path = "abfss://landing@stroutemindeuskadidev.dfs.core.windows.net/cultural/cultural_buildings.json"
delta_path = "abfss://bronze@stroutemindeuskadidev.dfs.core.windows.net/delta_tables/cultural_buildings/data"
deltaTable = "dbw_routemind_euskadi_dev.bronze.cultural_buildings"




df_qa_station_raw = spark.read \
    .option("multiline", "true") \
    .json(source_path)

# df_text = spark.read.format("text").option("wholetext", "true").load(source_path)

# file_content = df_text.select("value").collect()[0]["value"]

# rdd_text = df_text.select("value").rdd.map(lambda row: row.value)
# df_text_with_meta = df_text.withColumn("source_file_name", input_file_name())

# collected_rows = df_text_with_meta.collect()



In [0]:
raw_df = spark.read \
    .option("wholetext", "true") \
    .text(source_path)

In [0]:

content_cleaned = file_content.strip()
if content_cleaned.startswith('\ufeff'):
    content_cleaned = content_cleaned.encode('utf-8-sig').decode('utf-8')

# 4. Función para renombrar claves duplicadas (como los campos vacíos en address)[cite: 1] dinámicamente
def resolve_dups(pairs):
    d = {}
    for k, v in pairs:
        if k in d:
            new_k = f"{k}_dup"
            count = 1
            while new_k in d:
                count += 1
                new_k = f"{k}_dup_{count}"
            d[new_k] = v
        else:
            d[k] = v
    return d

# 5. Parsear el JSON completo de una sola vez utilizando el hook de duplicados
try:    parsed_json_list = json.loads(content_cleaned, object_pairs_hook=resolve_dups)
except json.JSONDecodeError as e:
    print(f"JSON decode error: {e}")
    # Optionally, attempt to fix common issues or load with a tolerant parser if available
    parsed_json_list = None


In [0]:
df_raw = spark.createDataFrame(processed_records)

In [0]:
def clean_json(text):
    # Hook personalizado: Evalúa las claves duplicadas mientras se construye el diccionario
    def remove_duplicates(json_key_value):
        clean_dict = {}
        for key, value in json_key_value:
            if key in clean_dict:
                # Si la clave ya existe pero el nuevo valor está vacío, conservamos el original
                if value == "" or value is None:
                    continue 
                # Si el valor original estaba vacío y el nuevo tiene datos, lo actualizamos
                elif clean_dict[key] == "" or clean_dict[key] is None:
                    clean_dict[key] = value
            else:
                clean_dict[key] = value
        return clean_dict
    
    try:
        # json.loads es tolerante a saltos de línea y permite aplicar nuestro hook
        parsed_data = json.loads(text, object_pairs_hook=remove_duplicates)
        return parsed_data
    except Exception as e:
        print(f"Parsing fail in file: {e}")
        return [] # Evita que el pipeline falle ante un error fatal, retornando lista vacía

In [0]:
rdd_cleaned_json = rdd_text.flatMap(process_json_duplicates)

# 5. Crear el DataFrame a partir de los JSON limpios
df_raw = spark.read.json(rdd_cleaned_json)

In [0]:
rdd_clean_data = rdd_text.flatMap(lambda file: clean_json(file))

# 5. Convertir cada diccionario limpio de vuelta a un string JSON individual
rdd_json = rdd_clean_data.map(lambda obj: json.dumps(obj))

# 6. Construir el DataFrame delegando la inferencia del esquema a Spark
# df_buildings_raw = spark.read.json(rdd_json)

print(rdd_json)

In [0]:
df_building_metadata = df_buildings_raw.withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_file", input_file_name())

In [0]:


df_qa_station_raw.write.option("path", delta_path).option("mergeSchema", True).saveAsTable(name=deltaTable, format="delta", mode="overwrite")
spark.sql(f"CREATE TABLE IF NOT EXISTS dbw_routemind_euskadi_dev.bronze.cultural_buildings USING DELTA LOCATION '{delta_path}'")
